## 1. Import Libraries

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

## 2. Load Dataset

In [2]:
fake_df=pd.read_csv('Fake.csv')
true_df=pd.read_csv('True.csv')

fake_df['label']='FAKE'
true_df['label']='REAL'

fake_df=fake_df[['title','text','label']]
true_df=true_df[['title','text','label']]

In [3]:
previous_df=pd.read_csv('fake_or_real_news.csv')

previous_df=previous_df[['title','text','label']]

In [4]:
from sklearn.utils import shuffle
df=pd.concat([fake_df,true_df,previous_df],ignore_index=True)
df = shuffle(df, random_state=42)

## 3. Data Cleaning

In [5]:
df.columns

Index(['title', 'text', 'label'], dtype='object')

In [6]:
df.shape

(51233, 3)

In [7]:
df.head()

,title,text,label
5952,Trump campaign woes intensify amid questions o...,Republicans face a lot of difficult decisions ...,REAL
14160,#AnyoneButHillary: NEW POLL Shows Bernie Suppo...,Hillary may find out she needs more than black...,FAKE
20775,Gruesome Uganda murders put police role in the...,KAMPALA (Reuters) - A spate of unsolved murder...,TRUE
5268,First planned North Korea-U.S. contact in Trum...,SEOUL (Reuters) - Plans for the first contact ...,TRUE
15978,EU MPs threaten Cambodia aid and trade over cr...,PHNOM PENH (Reuters) - A European Parliament d...,TRUE


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 51233 entries, 5952 to 15795
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   title   51233 non-null  object
 1   text    51233 non-null  object
 2   label   51233 non-null  object
dtypes: object(3)
memory usage: 1.6+ MB


In [9]:
df.isnull().sum()

,0
title,0
text,0
label,0


In [10]:
df[['title','text']].apply(lambda x:x.str.strip().eq("").sum())

,0
title,0
text,667


In [11]:
df[df['text'].str.strip().eq("")]

,title,text,label
12838,WOW! HOW 100% OF CLINTON CHARITY Went To…Thems...,,FAKE
11900,WATCH Life Accordion To Trump! [VIDEO],,FAKE
12370,SHOCKER! IS MITT ROMNEY Being Considered For A...,,FAKE
19783,BOYCOTT THE MEDIA! You Know It’s Bad When MSNB...,,FAKE
12038,WHERE’S HILLARY? CLINTON SPOTTED Dining Alone,,FAKE
...,...,...,...
12039,TRUMP ANNOUNCES Two More Companies Join “Ameri...,,FAKE
19115,TUCKER CARLSON Defends Trump On Sweden Comment...,,FAKE
19118,AWESOME! HISPANIC TRUMP SUPPORTER Rips Into Pr...,,FAKE
19769,DEMOCRAT OPERATIVES Caught Planning To Bully W...,,FAKE


In [12]:
df.drop(df[df['text'].str.strip().eq("")].index,inplace=True)

In [13]:
df.duplicated().sum()

np.int64(5604)

In [14]:
df.drop_duplicates(inplace=True)

## 4. Text Preprocessing

In [15]:
df["label"] = df["label"].replace("TRUE","REAL")

In [16]:
df['context']=df['title']+' '+df['text']

In [17]:
df['label'].value_counts()

,count
label,
REAL,23734
FAKE,20541


In [18]:
df['context']=df['context'].apply(lambda x: x.lower())

In [19]:
import re
df['context']=df['context'].apply(lambda x: re.sub(r"http\S+|www\S+","",x))

In [20]:
df['context']=df['context'].apply(lambda x: re.sub(r'[^a-zA-Z0-9\s]',"",x))

In [21]:
import nltk
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [22]:
from nltk.tokenize import word_tokenize
df['context']=df['context'].apply(word_tokenize)

In [23]:
import nltk
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


True

In [24]:
from nltk.corpus import stopwords
stop_words=stopwords.words('english')

In [25]:
def remove_stop_words(txt):
  t=[]
  for i in txt:
    if i not in stop_words:
      t.append(i)
  return ' '.join(t)
df['context']=df['context'].apply(remove_stop_words)


In [26]:
from nltk.stem import WordNetLemmatizer
lemma=WordNetLemmatizer()
def lemmatize_text(txt):
  word=word_tokenize(txt)
  t=[]
  for i in word:
    t.append(lemma.lemmatize(i))
  return " ".join(t)

df['context']=df['context'].apply(lemmatize_text)


In [27]:
df.head()

,title,text,label,context
5952,Trump campaign woes intensify amid questions o...,Republicans face a lot of difficult decisions ...,REAL,trump campaign woe intensify amid question mel...
14160,#AnyoneButHillary: NEW POLL Shows Bernie Suppo...,Hillary may find out she needs more than black...,FAKE,anyonebuthillary new poll show bernie supporte...
20775,Gruesome Uganda murders put police role in the...,KAMPALA (Reuters) - A spate of unsolved murder...,REAL,gruesome uganda murder put police role public ...
5268,First planned North Korea-U.S. contact in Trum...,SEOUL (Reuters) - Plans for the first contact ...,REAL,first planned north koreaus contact trump admi...
15978,EU MPs threaten Cambodia aid and trade over cr...,PHNOM PENH (Reuters) - A European Parliament d...,REAL,eu mp threaten cambodia aid trade crackdown ph...


In [28]:
def preprocess_text(txt):
  txt=txt.lower()
  txt=re.sub(r"http\S+|www\S+","",txt)
  txt=re.sub(r'[^a-zA-Z0-9\s]',"",txt)
  txt=word_tokenize(txt)    #--->(use this inside only if you not defined it outside)
  stop_words=stopwords.words('english')
  txt=[i for i in txt if i not in stop_words]
  lemma=WordNetLemmatizer()
  txt=[lemma.lemmatize(i) for i in txt]
  return " ".join(txt)

## 5. Label Encoding

In [29]:
from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()
df['label']=le.fit_transform(df['label'])

In [30]:
df['label'].value_counts()

,count
label,
1,23734
0,20541


## 6. Train-Test Split

In [31]:
X=df['context']
y=df['label']

In [32]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

## 7. Feature Extraction

In [33]:
from sklearn.feature_extraction.text import CountVectorizer , TfidfVectorizer

### Bag of Words

In [34]:
bow=CountVectorizer()
X_train_bow=bow.fit_transform(X_train)
X_test_bow=bow.transform(X_test)

In [35]:
X_train_bow.shape, X_test_bow.shape

((35420, 214967), (8855, 214967))

### TF-IDF

In [36]:
tfidf=TfidfVectorizer()
X_train_tfidf=tfidf.fit_transform(X_train)
X_test_tfidf=tfidf.transform(X_test)

In [37]:
X_train_tfidf.shape, X_test_tfidf.shape

((35420, 214967), (8855, 214967))

## 8. Model Training and Evaluation

In [38]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [39]:
lg=LogisticRegression()
lg.fit(X_train_bow, y_train)

LogisticRegression()

In [40]:
print(f'accuracy of lg-bow= {accuracy_score(y_test,lg.predict(X_test_bow))}')
print(f'accuracy on train={accuracy_score(y_train,lg.predict(X_train_bow))}')
print(f'classification report= \n{classification_report(y_test,lg.predict(X_test_bow))}')
print(f'confusion matrix lg-bow=\n{confusion_matrix(y_test,lg.predict(X_test_bow))}')

accuracy of lg-bow= 0.9734613212874083
accuracy on train=0.999802371541502
classification report= 
              precision    recall  f1-score   support

           0       0.97      0.97      0.97      4108
           1       0.98      0.97      0.98      4747

    accuracy                           0.97      8855
   macro avg       0.97      0.97      0.97      8855
weighted avg       0.97      0.97      0.97      8855

confusion matrix lg-bow=
[[4001  107]
 [ 128 4619]]


In [41]:
lg2=LogisticRegression()
lg2.fit(X_train_tfidf,y_train)

LogisticRegression()

In [42]:
print(f'accuracy of lg-tfidf= {accuracy_score(y_test,lg2.predict(X_test_tfidf))}')
print(f'accuracy on train={accuracy_score(y_train,lg2.predict(X_train_tfidf))}')
print(f'classification report= \n{classification_report(y_test,lg2.predict(X_test_tfidf))}')
print(f'confusion matrix lg-tfidf=\n{confusion_matrix(y_test,lg2.predict(X_test_tfidf))}')

accuracy of lg-tfidf= 0.9643139469226426
accuracy on train=0.97656691134952
classification report= 
              precision    recall  f1-score   support

           0       0.96      0.96      0.96      4108
           1       0.97      0.97      0.97      4747

    accuracy                           0.96      8855
   macro avg       0.96      0.96      0.96      8855
weighted avg       0.96      0.96      0.96      8855

confusion matrix lg-tfidf=
[[3946  162]
 [ 154 4593]]


In [43]:
svc=LinearSVC()
svc.fit(X_train_bow,y_train)

/usr/local/lib/python3.13/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


LinearSVC()

In [44]:
print(f'accuracy of svc-bow= {accuracy_score(y_test,svc.predict(X_test_bow))}')
print(f'accuracy on train={accuracy_score(y_train,svc.predict(X_train_bow))}')
print(f'classification report= \n{classification_report(y_test,svc.predict(X_test_bow))}')
print(f'confusion matrix svc-bow=\n{confusion_matrix(y_test,svc.predict(X_test_bow))}')

accuracy of svc-bow= 0.9673630717108977
accuracy on train=1.0
classification report= 
              precision    recall  f1-score   support

           0       0.96      0.97      0.96      4108
           1       0.97      0.97      0.97      4747

    accuracy                           0.97      8855
   macro avg       0.97      0.97      0.97      8855
weighted avg       0.97      0.97      0.97      8855

confusion matrix svc-bow=
[[3970  138]
 [ 151 4596]]


In [45]:
svc2=LinearSVC()
svc2.fit(X_train_tfidf,y_train)

LinearSVC()

In [46]:
print(f'accuracy of svc-tfidf= {accuracy_score(y_test,svc2.predict(X_test_tfidf))}')
print(f'accuracy on train={accuracy_score(y_train,svc2.predict(X_train_tfidf))}')
print(f'classification report= \n{classification_report(y_test,svc2.predict(X_test_tfidf))}')
print(f'confusion matrix svc-tfidf=\n{confusion_matrix(y_test,svc2.predict(X_test_tfidf))}')

accuracy of svc-tfidf= 0.976962168266516
accuracy on train=0.9988142292490119
classification report= 
              precision    recall  f1-score   support

           0       0.97      0.98      0.98      4108
           1       0.98      0.98      0.98      4747

    accuracy                           0.98      8855
   macro avg       0.98      0.98      0.98      8855
weighted avg       0.98      0.98      0.98      8855

confusion matrix svc-tfidf=
[[4009   99]
 [ 105 4642]]


In [47]:
nb=MultinomialNB()
nb.fit(X_train_bow,y_train)

MultinomialNB()

In [48]:
print(f'accuracy of nb-bow= {accuracy_score(y_test,nb.predict(X_test_bow))}')
print(f'accuracy on train={accuracy_score(y_train,nb.predict(X_train_bow))}')
print(f'classification report= \n{classification_report(y_test,nb.predict(X_test_bow))}')
print(f'confusion matrix nb-bow=\n{confusion_matrix(y_test,nb.predict(X_test_bow))}')

accuracy of nb-bow= 0.9247882552230379
accuracy on train=0.9432241671372106
classification report= 
              precision    recall  f1-score   support

           0       0.93      0.91      0.92      4108
           1       0.92      0.94      0.93      4747

    accuracy                           0.92      8855
   macro avg       0.92      0.92      0.92      8855
weighted avg       0.92      0.92      0.92      8855

confusion matrix nb-bow=
[[3742  366]
 [ 300 4447]]


In [49]:
nb2=MultinomialNB()
nb2.fit(X_train_tfidf,y_train)

MultinomialNB()

In [50]:
print(f'accuracy of nb-tfidf= {accuracy_score(y_test,nb2.predict(X_test_tfidf))}')
print(f'accuracy on train={accuracy_score(y_train,nb2.predict(X_train_tfidf))}')
print(f'classification report= \n{classification_report(y_test,nb2.predict(X_test_tfidf))}')
print(f'confusion matrix nb-tfidf=\n{confusion_matrix(y_test,nb2.predict(X_test_tfidf))}')

accuracy of nb-tfidf= 0.8928289102202146
accuracy on train=0.9076510446075663
classification report= 
              precision    recall  f1-score   support

           0       0.94      0.82      0.88      4108
           1       0.86      0.96      0.91      4747

    accuracy                           0.89      8855
   macro avg       0.90      0.89      0.89      8855
weighted avg       0.90      0.89      0.89      8855

confusion matrix nb-tfidf=
[[3358  750]
 [ 199 4548]]


In [51]:
import pandas as pd

results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Logistic Regression",
        "Multinomial Naive Bayes",
        "Multinomial Naive Bayes",
        "Linear SVM",
        "Linear SVM"
    ],
    "Feature Extraction": [
        "Bag of Words",
        "TF-IDF",
        "Bag of Words",
        "TF-IDF",
        "Bag of Words",
        "TF-IDF"
    ],
    "Train Accuracy": [
        0.9998,
        0.9765,
        0.9432,
        0.9076,
        1.0000,
        0.9988
    ],
    "Test Accuracy": [
        0.9734,
        0.9643,
        0.9247,
        0.8928,
        0.9672,
        0.9769
    ]
})

results.sort_values(by="Test Accuracy", ascending=False)

,Model,Feature Extraction,Train Accuracy,Test Accuracy
5,Linear SVM,TF-IDF,0.9988,0.9769
0,Logistic Regression,Bag of Words,0.9998,0.9734
4,Linear SVM,Bag of Words,1.0000,0.9672
1,Logistic Regression,TF-IDF,0.9765,0.9643
2,Multinomial Naive Bayes,Bag of Words,0.9432,0.9247
3,Multinomial Naive Bayes,TF-IDF,0.9076,0.8928


## 9. Hyperparameter Tuning

In [52]:
c_values=[0.01, 0.1, 1,2,5, 10,20]
for i in c_values:
  lg3=LogisticRegression(C=i)
  lg3.fit(X_train_bow,y_train)
  test_pred = lg3.predict(X_test_bow)
  train_pred = lg3.predict(X_train_bow)
  print(f"C={i}")
  print(f'accuracy of lg bow= {accuracy_score(y_test,test_pred)}')
  print(f'accuracy on train={accuracy_score(y_train,train_pred)}')
  print(f'classification report= \n{classification_report(y_test,test_pred)}')
  print(f'confusion matrix lg bow=\n{confusion_matrix(y_test,test_pred)}\n')

C=0.01
accuracy of lg bow= 0.9738001129305477
accuracy on train=0.9863918690005646
classification report= 
              precision    recall  f1-score   support

           0       0.96      0.98      0.97      4108
           1       0.98      0.97      0.98      4747

    accuracy                           0.97      8855
   macro avg       0.97      0.97      0.97      8855
weighted avg       0.97      0.97      0.97      8855

confusion matrix lg bow=
[[4029   79]
 [ 153 4594]]

C=0.1
accuracy of lg bow= 0.9770750988142293
accuracy on train=0.9973461321287408
classification report= 
              precision    recall  f1-score   support

           0       0.97      0.98      0.98      4108
           1       0.98      0.98      0.98      4747

    accuracy                           0.98      8855
   macro avg       0.98      0.98      0.98      8855
weighted avg       0.98      0.98      0.98      8855

confusion matrix lg bow=
[[4023   85]
 [ 118 4629]]

C=1
accuracy of lg bow= 0.9

In [53]:
c_values=[0.01, 0.1, 1,2,5, 10,20]
for i in c_values:
  svc3=LinearSVC(C=i)
  svc3.fit(X_train_tfidf,y_train)
  test_pred = svc3.predict(X_test_tfidf)
  train_pred = svc3.predict(X_train_tfidf)
  print(f"C={i}")
  print(f'accuracy of linear svc-tfidf= {accuracy_score(y_test,test_pred)}')
  print(f'accuracy on train={accuracy_score(y_train,train_pred)}')
  print(f'classification report= \n{classification_report(y_test,test_pred)}')
  print(f'confusion matrix linear svc-tfidf=\n{confusion_matrix(y_test,test_pred)}\n')

C=0.01
accuracy of linear svc-tfidf= 0.9425183512140034
accuracy on train=0.9470073404856013
classification report= 
              precision    recall  f1-score   support

           0       0.94      0.94      0.94      4108
           1       0.95      0.95      0.95      4747

    accuracy                           0.94      8855
   macro avg       0.94      0.94      0.94      8855
weighted avg       0.94      0.94      0.94      8855

confusion matrix linear svc-tfidf=
[[3858  250]
 [ 259 4488]]

C=0.1
accuracy of linear svc-tfidf= 0.9681535855448898
accuracy on train=0.9803500846979107
classification report= 
              precision    recall  f1-score   support

           0       0.96      0.97      0.97      4108
           1       0.97      0.97      0.97      4747

    accuracy                           0.97      8855
   macro avg       0.97      0.97      0.97      8855
weighted avg       0.97      0.97      0.97      8855

confusion matrix linear svc-tfidf=
[[3972  136]
 [

### **Hyperparameter Tuning Result**

For Logistic Regression with Bag-of-Words, C=0.1 produced the highest
test accuracy of 97.71%.

For LinearSVC with TF-IDF, C=1 produced a test accuracy of 97.70%.
Although Logistic Regression + BOW achieved a marginally higher
held-out test accuracy, LinearSVC + TF-IDF was selected for the final
model based on its performance on the independent external dataset.

## 10. Final Model

In [54]:
final_lg_model = LogisticRegression(C=0.1)
final_lg_model.fit(X_train_bow, y_train)
lg_test=final_lg_model.predict(X_test_bow)
lg_train=final_lg_model.predict(X_train_bow)
print(f'accuracy of lg bow= {accuracy_score(y_test,lg_test)}')
print(f'accuracy on train={accuracy_score(y_train,lg_train)}')
print(f'classification report= \n{classification_report(y_test,lg_test)}')
print(f'confusion matrix lg bow=\n{confusion_matrix(y_test,lg_test)}\n')

accuracy of lg bow= 0.9770750988142293
accuracy on train=0.9973461321287408
classification report= 
              precision    recall  f1-score   support

           0       0.97      0.98      0.98      4108
           1       0.98      0.98      0.98      4747

    accuracy                           0.98      8855
   macro avg       0.98      0.98      0.98      8855
weighted avg       0.98      0.98      0.98      8855

confusion matrix lg bow=
[[4023   85]
 [ 118 4629]]



In [55]:
final_svc_model=LinearSVC(C=1)
final_svc_model.fit(X_train_tfidf,y_train)
svc_test=final_svc_model.predict(X_test_tfidf)
svc_train=final_svc_model.predict(X_train_tfidf)
print(f'accuracy of svc-tfidf= {accuracy_score(y_test,svc_test)}')
print(f'accuracy on train={accuracy_score(y_train,svc_train)}')
print(f'classification report= \n{classification_report(y_test,svc_test)}')
print(f'confusion matrix svc-tfidf=\n{confusion_matrix(y_test,svc_test)}\n')

accuracy of svc-tfidf= 0.976962168266516
accuracy on train=0.9988142292490119
classification report= 
              precision    recall  f1-score   support

           0       0.97      0.98      0.98      4108
           1       0.98      0.98      0.98      4747

    accuracy                           0.98      8855
   macro avg       0.98      0.98      0.98      8855
weighted avg       0.98      0.98      0.98      8855

confusion matrix svc-tfidf=
[[4009   99]
 [ 105 4642]]



## 11. Save Model and Vectorizer

In [56]:
import joblib

In [57]:
joblib.dump(bow,"bow_vectorizer.pkl")
joblib.dump(final_lg_model,"final_lg_model.pkl")

['final_lg_model.pkl']

In [58]:
joblib.dump(tfidf,"tfidf_vectorizer.pkl")
joblib.dump(final_svc_model,"final_svc_model.pkl")

['final_svc_model.pkl']

## 12. Test Prediction Pipeline

In [59]:
loaded_bow = joblib.load("bow_vectorizer.pkl")
loaded_lg_model = joblib.load("final_lg_model.pkl")
sample_text = ["This is a sample news article for testing the model"]

sample_vector = loaded_bow.transform(sample_text)

prediction = loaded_lg_model.predict(sample_vector)

print(prediction)

[0]


In [60]:
loaded_tfidf = joblib.load("tfidf_vectorizer.pkl")
loaded_svc_model = joblib.load("final_svc_model.pkl")
sample_text = ["This is a sample news article for testing the model"]

sample_vector = loaded_tfidf.transform(sample_text)

prediction = loaded_svc_model.predict(sample_vector)

print(prediction)

[0]


In [61]:
sample_text = "Title: Government Announces Major Expansion of Electric Bus Network Article: The government announced Tuesday that it will introduce thousands of new electric buses across major cities over the next three years. According to officials, the program will focus on replacing older diesel buses, reducing emissions and improving public transportation. The transportation ministry said funding for the first phase has been approved, with new charging stations expected to be installed alongside the buses."

cleaned_text = preprocess_text(sample_text)

vectorized_text = loaded_bow.transform([cleaned_text])

prediction = loaded_lg_model.predict(vectorized_text)

print("Prediction:", le.inverse_transform(prediction)[0])

Prediction: FAKE


In [62]:
sample_text = "Title: Government Announces Major Expansion of Electric Bus Network Article: The government announced Tuesday that it will introduce thousands of new electric buses across major cities over the next three years. According to officials, the program will focus on replacing older diesel buses, reducing emissions and improving public transportation. The transportation ministry said funding for the first phase has been approved, with new charging stations expected to be installed alongside the buses."

cleaned_text = preprocess_text(sample_text)

vectorized_text = loaded_tfidf.transform([cleaned_text])

prediction = loaded_svc_model.predict(vectorized_text)

print("Prediction:", le.inverse_transform(prediction)[0])

Prediction: REAL


In [67]:
sample_text = """U.S. Secretary of State John F. Kerry said Monday that he will stop in Paris later this week, amid criticism that no top American officials attended Sunday’s unity march against terrorism.

Kerry said he expects to arrive in Paris Thursday evening, as he heads home after a week abroad. He said he will fly to France at the conclusion of a series of meetings scheduled for Thursday in Sofia, Bulgaria. He plans to meet the next day with Foreign Minister Laurent Fabius and President Francois Hollande, then return to Washington.

The visit by Kerry, who has family and childhood ties to the country and speaks fluent French, could address some of the criticism that the United States snubbed France in its darkest hour in many years.

The French press on Monday was filled with questions about why neither President Obama nor Kerry attended Sunday’s march, as about 40 leaders of other nations did."""

cleaned_text = preprocess_text(sample_text)

vectorized_text = loaded_bow.transform([cleaned_text])

prediction = loaded_lg_model.predict(vectorized_text)

print("Prediction:", le.inverse_transform(prediction)[0])

Prediction: REAL


In [68]:
sample_text = """U.S. Secretary of State John F. Kerry said Monday that he will stop in Paris later this week, amid criticism that no top American officials attended Sunday’s unity march against terrorism.

Kerry said he expects to arrive in Paris Thursday evening, as he heads home after a week abroad. He said he will fly to France at the conclusion of a series of meetings scheduled for Thursday in Sofia, Bulgaria. He plans to meet the next day with Foreign Minister Laurent Fabius and President Francois Hollande, then return to Washington.

The visit by Kerry, who has family and childhood ties to the country and speaks fluent French, could address some of the criticism that the United States snubbed France in its darkest hour in many years.

The French press on Monday was filled with questions about why neither President Obama nor Kerry attended Sunday’s march, as about 40 leaders of other nations did."""

cleaned_text = preprocess_text(sample_text)

vectorized_text = loaded_tfidf.transform([cleaned_text])

prediction = loaded_svc_model.predict(vectorized_text)

print("Prediction:", le.inverse_transform(prediction)[0])

Prediction: REAL


## 13. I**ndependent External Validation**

To evaluate whether the selected model generalizes beyond the data used
during training and testing, the final model was evaluated on a separate
external dataset containing 45,757 news articles.

The external dataset was not used during model training.

The LinearSVC + TF-IDF model achieved 97.41% accuracy on this dataset.

This evaluation measures performance on this particular external dataset
and should not be interpreted as real-world fact-checking accuracy.

In [63]:
n_df=pd.read_csv('news.csv')
n_df["label"].value_counts()

,count
label,
1,22900
0,22857


In [64]:
n_df.head()

,text,label
0,Gere faults Trump for blurring meaning of 'ref...,1
1,German parties start to find common ground in ...,1
2,Senate Democratic leader says Attorney General...,1
3,"Tennis: Kyrgios fined $10,000 for Shanghai wal...",1
4,Trump Threw Mar-A-Lago Fundraiser For Woman A...,0


In [65]:
n_df['text']=n_df['text'].apply(preprocess_text)
testing_data=loaded_bow.transform(n_df['text'])
prediction=loaded_lg_model.predict(testing_data)

print(f"accuracy ={accuracy_score(n_df['label'],prediction)}")
print(f"classification report= \n{classification_report(n_df['label'],prediction)}")
print(f"confusion matrix =\n{confusion_matrix(n_df['label'],prediction)}\n")

accuracy =0.973512249491881
classification report= 
              precision    recall  f1-score   support

           0       0.96      0.98      0.97     22857
           1       0.98      0.96      0.97     22900

    accuracy                           0.97     45757
   macro avg       0.97      0.97      0.97     45757
weighted avg       0.97      0.97      0.97     45757

confusion matrix =
[[22483   374]
 [  838 22062]]



In [66]:
n_df['text']=n_df['text'].apply(preprocess_text)
testing_data=loaded_tfidf.transform(n_df['text'])
prediction=loaded_svc_model.predict(testing_data)

print(f"accuracy ={accuracy_score(n_df['label'],prediction)}")
print(f"classification report= \n{classification_report(n_df['label'],prediction)}")
print(f"confusion matrix =\n{confusion_matrix(n_df['label'],prediction)}\n")

accuracy =0.97405861398256
classification report= 
              precision    recall  f1-score   support

           0       0.96      0.98      0.97     22857
           1       0.98      0.96      0.97     22900

    accuracy                           0.97     45757
   macro avg       0.97      0.97      0.97     45757
weighted avg       0.97      0.97      0.97     45757

confusion matrix =
[[22491   366]
 [  821 22079]]



### **Final Model Selection**

LinearSVC with TF-IDF and C=1 was selected as the final model.

The model achieved 97.70% accuracy on the held-out test set and 97.41%
accuracy on an independent external dataset containing 45,757 articles.

Logistic Regression with Bag-of-Words and C=0.1 achieved a very similar
performance, with 97.71% accuracy on the held-out test set and 97.35%
on the external dataset.

The final model was selected based on the combination of held-out and
independent external evaluation.